In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!cp -r /content/drive/MyDrive/FinalYear_Research/MonoPoly_Model/features /content/monopoly_data

In [9]:
FEATURE_ROOT = "/content/monopoly_data"

In [12]:
!ls /content/monopoly_data

test  train  val


In [6]:
!pip install torch torchvision torchaudio librosa numpy scikit-learn tqdm

In [10]:
import numpy as np
sample = np.load(
    next(iter(
        __import__("pathlib").Path(FEATURE_ROOT + "/train/mono").glob("*.npy")
    ))
)
print(sample.shape)

(128, 94)


In [13]:
import torch
from torch.utils.data import Dataset
from pathlib import Path
import numpy as np

class MonoPolyDataset(Dataset):
    def __init__(self, root, split):
        self.files = []
        self.labels = []

        for label, cls in enumerate(["mono", "poly"]):
            paths = Path(root, split, cls).glob("*.npy")
            for p in paths:
                self.files.append(p)
                self.labels.append(label)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        x = np.load(self.files[idx])
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(0)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

In [25]:
from torch.utils.data import DataLoader

train_ds = MonoPolyDataset(FEATURE_ROOT, "train")
val_ds   = MonoPolyDataset(FEATURE_ROOT, "val")
test_ds  = MonoPolyDataset(FEATURE_ROOT, "test")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)
test_loader  = DataLoader(test_ds, batch_size=32)

print(len(train_ds), len(val_ds), len(test_ds))

30800 6600 6600


In [ ]:
import torch.nn as nn

class CRNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )

        self.gru = nn.GRU(
            input_size=64 * 32,
            hidden_size=128,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(256, 2)

    def forward(self, x):
        x = self.cnn(x)               
        x = x.permute(0, 3, 1, 2)     # Reshape for GRU
        x = x.flatten(2)              
        x, _ = self.gru(x)
        x = x.mean(dim=1)
        return self.fc(x)

In [27]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CRNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [28]:
from tqdm import tqdm

def train_epoch(loader):
    model.train()
    total_loss, correct = 0, 0

    for x, y in tqdm(loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (out.argmax(1) == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

In [29]:
def eval_epoch(loader):
    model.eval()
    correct = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(1) == y).sum().item()

    return correct / len(loader.dataset)

In [30]:
EPOCHS = 25

for epoch in range(EPOCHS):
    loss, acc = train_epoch(train_loader)
    val_acc = eval_epoch(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f} | Train Acc: {acc:.3f} | Val Acc: {val_acc:.3f}")

100%|██████████| 963/963 [00:38<00:00, 25.05it/s]


Epoch 1/25 | Loss: 0.1680 | Train Acc: 0.942 | Val Acc: 0.947


100%|██████████| 963/963 [00:32<00:00, 29.96it/s]


Epoch 2/25 | Loss: 0.1161 | Train Acc: 0.962 | Val Acc: 0.957


100%|██████████| 963/963 [00:22<00:00, 43.49it/s]


...

Epoch 23/25 | Loss: 0.0225 | Train Acc: 0.993 | Val Acc: 0.972


100%|██████████| 963/963 [00:22<00:00, 43.34it/s]


Epoch 24/25 | Loss: 0.0175 | Train Acc: 0.994 | Val Acc: 0.969


100%|██████████| 963/963 [00:22<00:00, 43.46it/s]


Epoch 25/25 | Loss: 0.0176 | Train Acc: 0.995 | Val Acc: 0.971


In [31]:
test_acc = eval_epoch(test_loader)
print(f"Test Accuracy: {test_acc:.3f}")

Test Accuracy: 0.963


In [32]:
from sklearn.metrics import confusion_matrix
import numpy as np

y_true, y_pred = [], []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        y_true.extend(y.numpy())
        y_pred.extend(out.argmax(1).cpu().numpy())

print(confusion_matrix(y_true, y_pred))

[[3230   70]
 [ 175 3125]]


In [33]:
torch.save(model.state_dict(), "mono_poly_crnn.pth")

In [37]:
from google.colab import files
files.download("mono_poly_crnn.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>